# KSL Sign Language — Training Notebook

Trains `MultiHeadSignModel` on extracted MediaPipe landmarks. Supports a **two-stage**
pipeline via the `STAGE` toggle in §2:

1. `rsl_pretrain` — pretrain the shared GCN + Transformer encoder on **Slovo**
   (Russian Sign Language, ~1000 classes) for a richer initialization.
2. `ksl_finetune` — train/fine-tune the KSL head, optionally loading the
   Stage-1 encoder weights (`checkpoints/rsl_pretrain.pt`) via `PRETRAINED`.

**Pipeline:** `.npz` landmarks → augmentation → `KSLLandmarkDataset` → `PadCollate`
→ `SpatialGCN` (mask-aware, **residual**) → `TemporalTransformer` (CLS token) → classifier head

| Section | What it does |
|---|---|
| 1. Imports | Libraries + local modules |
| 2. Config | Stage toggle (`rsl_pretrain` / `ksl_finetune`) + all hyperparameters |
| 3. Device | Auto-select MPS / CUDA / CPU |
| 4. Data | Scan landmarks, split (or dedicated val dir) |
| 5. Augmentation | Configure the augmentation pipeline |
| 6. DataLoaders | Build train / val loaders |
| 7. Model | Build / load pretrained checkpoint |
| 8. Optimiser | AdamW + linear warmup + plateau scheduler |
| 9. Training | Epoch loop with best-checkpoint saving |
| 10. Results | Loss, accuracy, LR curves |

## 1. Imports

In [1]:
import sys
from pathlib import Path

_here = (Path("__file__").resolve().parent
         if "__file__" in dir() else Path(".").resolve())
sys.path.insert(0, str(_here))            # scripts/training  (this notebook's own dir)
sys.path.insert(0, str(_here.parent))     # scripts            (core modules live here)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

from augmentations import AugmentPipeline
from dataset import (KSLLandmarkDataset, scan_landmarks, scan_val_landmarks,
                      stratified_split, PadCollate)
from model_claude import MultiHeadSignModel, count_parameters

print("torch", torch.__version__)


torch 2.12.0


## 2. Configuration

Pick a `STAGE`, then all paths / LR / schedule hyperparameters below switch
automatically. Model architecture (`D_MODEL`, `NUM_LAYERS`, ...) stays shared
across stages so Stage-1 encoder weights load cleanly into Stage-2.

**Underfitting checklist** (in order of impact):

| Parameter | Anti-underfit direction | Value |
|---|---|---|
| `D_MODEL` | ↑ bigger backbone | **256** |
| `NUM_LAYERS` | ↑ more temporal depth | **6** |
| `DROPOUT` | ↓ less regularization | **0.25** |
| `WEIGHT_DECAY` | ↓ less L2 penalty | stage-dependent |
| `WARMUP_EPOCHS` | stable Transformer start | stage-dependent |
| `EPOCHS` | train longer | stage-dependent |
| `BATCH_SIZE` | smoother gradients | **32** |
| `USE_AUGMENTATION` | `False` to diagnose | True |
| `LABEL_SMOOTHING` | ↓ lower to reduce regularization | **0.1** |

> **Diagnosis tip:** if train acc stays near random (1%) for 5+ epochs, set
> `USE_AUGMENTATION = False` and re-run.  If it then overfits, augmentation was
> the bottleneck.  If it still can't learn, increase `D_MODEL` or lower `ENC_LR`.
>
> **Overfitting tip** (train acc pulling away from val acc, or val loss rising while
> val acc is flat): bump `DROPOUT` further, raise `WEIGHT_DECAY`, raise
> `LABEL_SMOOTHING`, and switch §5 to `AugmentPipeline.strong()` (already the default
> below) before shrinking the model.
>
> **Fine-tuning tip** (`ksl_finetune` loading a Stage-1 checkpoint): keep `ENC_LR`
> well below `HEAD_LR` (currently 1e-4 vs 1e-3) -- the encoder already has useful
> weights, a fresh head does not, so hitting them with the same LR risks partially
> erasing what Stage 1 learned.

In [2]:
# ── Stage selection ──────────────────────────────────────────────────────────
# "<language>_pretrain"   -- Stage 1, ONE language (e.g. "rsl_pretrain",
#                            "wlasl_pretrain", "arabic2_pretrain", ...) -- auto-
#                            generated below, one per key in LANGUAGES. Uses the
#                            original single-dataset path/log format (same as
#                            the old hand-written "rsl_pretrain" always did).
# "multilingual_pretrain" -- Stage 1, JOINT: pretrain on ALL active languages at
#                            once, round-robin interleaved, one head per language.
#                            Different log format (per-language line each epoch)
#                            -- mirrors scripts/training/train_multilingual.py.
# "ksl_finetune"          -- Stage 2: train/fine-tune the KSL head (optionally
#                            from any Stage-1 checkpoint above)
STAGE = "multilingual_pretrain"

# Every language extracted so far. Used by ALL "<language>_pretrain" stages
# (each just needs its own entry) AND by "multilingual_pretrain" (uses all of
# them at once). A "data" path that doesn't exist on disk yet only breaks
# things if you actually select THAT language's stage -- others are unaffected.
LANGUAGES = {
    # "rsl":       dict(data="../../data/processed_landmarks/slovo/train", val_data="../../data/processed_landmarks/slovo/test"),
    # "arabic":    dict(data="../../data/processed_landmarks/arabic",   val_data=None),  # was "arabic" -- no such dir, actual extract is "arabic2"
    "ksl":       dict(data="../../data/processed_landmarks/ksl",       val_data=None),  # co-training: KSL joins the round-robin
    "wlasl":     dict(data="../../data/processed_landmarks/wlasl",     val_data=None),
    # "indian":   dict(data="../../data/processed_landmarks/indian",   val_data=None),
    # "srilanka":  dict(data="../../data/processed_landmarks/srilanka",  val_data=None),
    # "argentina": dict(data="../../data/processed_landmarks/argentina", val_data=None),
    # "vietnam": dict(data="../../data/processed_landmarks/vietnam", val_data=None),
}
STEPS_PER_LANG = 40   # batches drawn from EACH active language per epoch (multilingual only)

# Shared Stage-1 hyperparameters -- same tuned values rsl_pretrain converged on
# (see DRAFT_NOTES.md's overfitting investigation: label smoothing + weight
# decay + dropout bump). Every single-language "<lang>_pretrain" stage AND
# "multilingual_pretrain" use these; ksl_finetune has its own (below).
_PRETRAIN_DEFAULTS = dict(
    EPOCHS=500, ENC_LR=5e-4, HEAD_LR=1e-3, WEIGHT_DECAY=4e-4,
    WARMUP_EPOCHS=10, SCHED_PATIENCE=12, LABEL_SMOOTHING=0.1,
)

_ALL_STAGES = {
    f"{lang}_pretrain": dict(
        DATA_DIR=cfg["data"], VAL_DIR=cfg.get("val_data"),
        OUT_PATH=f"../../checkpoints/{lang}_pretrain.pt",
        HEAD=lang, PRETRAINED=None,
        **_PRETRAIN_DEFAULTS,
    )
    for lang, cfg in LANGUAGES.items()
}
_ALL_STAGES["multilingual_pretrain"] = dict(
    DATA_DIR       = None,          # not used -- see LANGUAGES above
    VAL_DIR        = None,          # not used -- per-language val_data / stratified split
    OUT_PATH       = "../../checkpoints/multilingual_pretrain.pt",
    HEAD           = None,          # not used -- one head per language, see §7
    PRETRAINED     = None,
    **_PRETRAIN_DEFAULTS,
)
_ALL_STAGES["ksl_finetune"] = dict(
    DATA_DIR       = "../../data/processed_landmarks/ksl",
    VAL_DIR        = None,               # None -> stratified split of DATA_DIR
    OUT_PATH       = "../../checkpoints/ksl_best.pt",
    HEAD           = "ksl",
    PRETRAINED     = "/Users/macbook/Desktop/KSL Related Project/KSL Project/Pretrained Models/RSL_Pretrain/models/rsl_pretrain_e140_acc0.6712.pt",  # swap to any other
                                                             # *_pretrain.pt above
    EPOCHS         = 170,
    ENC_LR         = 1e-4,   # << pretrained encoder needs a much smaller LR than
                             #    the fresh head, or fine-tuning partially erases
                             #    what Stage 1 learned
    HEAD_LR        = 1e-3,
    WEIGHT_DECAY   = 3e-5,
    WARMUP_EPOCHS  = 10,
    SCHED_PATIENCE = 8,
    LABEL_SMOOTHING = 0.1,
)

_STAGE_CFG = _ALL_STAGES[STAGE]

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR    = _STAGE_CFG["DATA_DIR"]     # unused when STAGE == "multilingual_pretrain"
VAL_DIR     = _STAGE_CFG["VAL_DIR"]      # dedicated val root, or None to split DATA_DIR
OUT_PATH    = _STAGE_CFG["OUT_PATH"]
PRETRAINED  = _STAGE_CFG["PRETRAINED"]   # Stage-1 checkpoint to warm-start the encoder

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS         = _STAGE_CFG["EPOCHS"]
BATCH_SIZE     = 32
MAX_LEN        = 128
VAL_FRAC       = 0.15 if STAGE == "multilingual_pretrain" else 0.2  # languages w/o a dedicated val split
SEED           = 67
WORKERS        = 2 if STAGE == "multilingual_pretrain" else 4  # x N active languages -> total processes

# ── Optimiser & schedule ─────────────────────────────────────────────────────
ENC_LR         = _STAGE_CFG["ENC_LR"]
HEAD_LR        = _STAGE_CFG["HEAD_LR"]
WEIGHT_DECAY   = _STAGE_CFG["WEIGHT_DECAY"]
WARMUP_EPOCHS  = _STAGE_CFG["WARMUP_EPOCHS"]
SCHED_PATIENCE = _STAGE_CFG["SCHED_PATIENCE"]
LABEL_SMOOTHING = _STAGE_CFG["LABEL_SMOOTHING"]  # train loss only; val loss stays plain CE

# ── Model architecture (shared across stages) ────────────────────────────────
IN_CH      = 5           # x, y, dx, dy, vis
D_MODEL    = 256
GCN_HIDDEN = (128, 256)
NHEAD      = 8
NUM_LAYERS = 6
DIM_FF     = 512
DROPOUT    = 0.25   # << was 0.2; train/val gap + rising val loss on rsl_pretrain called for more

HEAD = _STAGE_CFG["HEAD"]        # unused when STAGE == "multilingual_pretrain"

print(f"STAGE={STAGE}  data={DATA_DIR}  val={VAL_DIR}  head={HEAD}  "
      f"pretrained={PRETRAINED}  out={OUT_PATH}")
if STAGE == "multilingual_pretrain":
    print(f"languages registered: {list(LANGUAGES)}  |  steps/lang/epoch={STEPS_PER_LANG}")


STAGE=multilingual_pretrain  data=None  val=None  head=None  pretrained=None  out=../../checkpoints/multilingual_pretrain.pt
languages registered: ['ksl', 'wlasl']  |  steps/lang/epoch=40


## 3. Device & Reproducibility

In [3]:
def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")    # Apple Silicon GPU
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

torch.manual_seed(SEED)
device = pick_device()
print("device:", device)

device: mps


## 4. Data Loading

Expected layout (same for both stages — `<class>/<clip>.npz`):
```
data/processed_landmarks/ksl/            data/processed_landmarks/slovo/
    ចង់/   clip_001.npz        train/  <class>/  *.npz
    ចូល/   ...                 test/   <class>/  *.npz
    ...
```
Each `.npz` must contain `coords (T, 75, 2)` and `vis (T, 75)`.

If `VAL_DIR` is set (Slovo's `rsl_pretrain` stage), it's scanned as a
**dedicated** validation set via `scan_val_landmarks` — no split needed.
Otherwise (KSL's `ksl_finetune` stage) `DATA_DIR` is stratified-split by `VAL_FRAC`.

In [4]:
if STAGE == "multilingual_pretrain":
    def _load_language(name, cfg):
        data_dir = Path(cfg["data"])
        if not data_dir.exists():
            print(f"  [{name}] SKIPPED -- {data_dir} not found (not extracted yet?)")
            return None
        items, class_to_idx = scan_landmarks(data_dir)
        if cfg.get("val_data"):
            train_items = items
            val_items = scan_val_landmarks(cfg["val_data"], class_to_idx)
        else:
            train_items, val_items = stratified_split(items, VAL_FRAC, SEED)
        print(f"  [{name:10s}] {len(class_to_idx):4d} classes | "
              f"{len(train_items):6d} train / {len(val_items):5d} val")
        return dict(class_to_idx=class_to_idx, num_classes=len(class_to_idx),
                    train_items=train_items, val_items=val_items)

    print(f"Scanning {len(LANGUAGES)} registered language(s):")
    langs = {}
    for name, cfg in sorted(LANGUAGES.items()):
        result = _load_language(name, cfg)
        if result is not None:
            langs[name] = result
    if not langs:
        raise RuntimeError("No languages available -- extract at least one before running this stage.")
    print(f"\n{len(langs)} active language(s): {sorted(langs)}")

else:
    items, class_to_idx = scan_landmarks(DATA_DIR)
    num_classes = len(class_to_idx)

    if VAL_DIR:
        train_items = items
        val_items = scan_val_landmarks(VAL_DIR, class_to_idx)
        print(f"{len(train_items)} train clips (from {DATA_DIR})  |  "
              f"{len(val_items)} val clips (from {VAL_DIR})  |  {num_classes} classes")
    else:
        train_items, val_items = stratified_split(items, VAL_FRAC, SEED)
        print(f"{len(items)} clips  |  {num_classes} classes")
        print(f"train: {len(train_items)}  |  val: {len(val_items)}")

    print("classes:", list(class_to_idx.keys())[:12],
          "..." if num_classes > 12 else "")


Scanning 2 registered language(s):
scan_landmarks: skipped 234 clip(s) under 3 frames
  [ksl       ]  194 classes |   4104 train /   714 val
  [wlasl     ] 2000 classes |  10371 train /  2134 val

2 active language(s): ['ksl', 'wlasl']


## 5. Augmentation

| Transform | Effect |
|---|---|
| `temporal_crop` | Random sub-sequence ≥ 70% of T |
| `speed_perturb` | Resample at 0.7×–1.3× speed |
| `temporal_jitter` | Drop random frames |
| `horizontal_flip` | Mirror + swap L/R hands & pose pairs |
| `rotation_2d` | Rotate ±12° in image plane |
| `anchor_jitter` | Perturb shoulder (normalization) anchors |
| `reach_scale` | Scale arm/hand reach from body centre |
| `gaussian_noise` | Per-joint i.i.d. noise on visible landmarks |
| `joint_masking` | Zero random joints (same joints all frames) |
| `frame_masking` | Zero random entire frames |

**If still underfitting:** set `USE_AUGMENTATION = False` first to confirm the
model can overfit training data.  If it can → re-enable augmentation and train
longer.  If it can't overfit at all → increase `D_MODEL` / `NUM_LAYERS`.

In [5]:
USE_AUGMENTATION = True

# ── Choose a pipeline ────────────────────────────────────────────────────────
# AugmentPipeline.default()  -- full pipeline, ~5-6 transforms fire per sample
# AugmentPipeline.light()    -- lighter, better for small datasets or fine-tuning
# AugmentPipeline.strong()   -- heavier probs/magnitudes; use when train/val gap widens
# AugmentPipeline([...])     -- build a custom pipeline from individual transforms

augmenter = AugmentPipeline.strong() if USE_AUGMENTATION else None

if augmenter is not None:
    print(f"augmentation ON  ({len(augmenter.transforms)} transforms)")
    for fn, prob, kwargs in augmenter.transforms:
        kw_str = "  " + ", ".join(f"{k}={v}" for k, v in kwargs.items()) if kwargs else ""
        print(f"  p={prob:.1f}  {fn.__name__}{kw_str}")
else:
    print("augmentation OFF")

augmentation ON  (10 transforms)
  p=0.8  temporal_crop  min_frac=0.6
  p=0.6  speed_perturb  min_speed=0.6, max_speed=1.4
  p=0.4  temporal_jitter  drop_prob=0.08
  p=0.5  horizontal_flip
  p=0.6  rotation_2d  max_deg=18.0
  p=0.6  anchor_jitter  max_shift=0.035
  p=0.5  reach_scale  min_scale=0.82, max_scale=1.18
  p=0.8  gaussian_noise  std=0.008
  p=0.5  joint_masking  p=0.15
  p=0.4  frame_masking  p=0.15


In [6]:
collate = PadCollate(use_velocity=True)

# drop_last=True on EVERY loader (train and val, both branches): with
# DataParallel across 2 GPUs, a leftover under-sized last batch gets split
# unevenly across replicas (or produces an empty/degenerate shard on one of
# them), and the fused Transformer-encoder-layer CUDA kernel chokes on that
# ("misaligned address" on the smaller replica -- reproduced on Kaggle's T4x2).
# Losing at most BATCH_SIZE-1 samples' worth of val precision per epoch is a
# fine trade for not crashing mid-training.
if STAGE == "multilingual_pretrain":
    # Augmentation is applied only to the training set; val always uses clean data.
    for name, lang in langs.items():
        train_ds = KSLLandmarkDataset(lang["train_items"], lang["class_to_idx"], MAX_LEN, augment=augmenter)
        val_ds   = KSLLandmarkDataset(lang["val_items"],   lang["class_to_idx"], MAX_LEN)
        lang["train_loader"] = DataLoader(
            train_ds, batch_size=BATCH_SIZE, shuffle=True,
            collate_fn=collate, num_workers=WORKERS, drop_last=True,
        )
        lang["val_loader"] = DataLoader(
            val_ds, batch_size=BATCH_SIZE, shuffle=False,
            collate_fn=collate, num_workers=WORKERS, drop_last=True,
        )
        print(f"  [{name:10s}] {len(lang['train_loader']):4d} train batches | "
              f"{len(lang['val_loader']):4d} val batches")

else:
    # Augmentation is applied only to the training set; val always uses clean data.
    train_ds = KSLLandmarkDataset(train_items, class_to_idx, MAX_LEN, augment=augmenter)
    val_ds   = KSLLandmarkDataset(val_items,   class_to_idx, MAX_LEN)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate, num_workers=WORKERS, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate, num_workers=WORKERS, drop_last=True,
    )

    print(f"train batches: {len(train_loader)}  |  val batches: {len(val_loader)}")


  [ksl       ]  128 train batches |   22 val batches
  [wlasl     ]  324 train batches |   66 val batches


## 6. DataLoaders

## 7. Model

`MultiHeadSignModel` = `SpatialGCN` (mask-aware **+** residual skip connections)
+ `TemporalTransformer` (CLS token) + swappable head.

The head is created for the current `STAGE`'s `HEAD` name (`"rsl"` or `"ksl"`)
with `num_classes` matching that stage's dataset. When `PRETRAINED` points at a
Stage-1 checkpoint, `load_state_dict(..., strict=False)` loads the shared
encoder weights and leaves the (stage-specific) head randomly initialized.

At `D_MODEL=256` / `NUM_LAYERS=6` the backbone is ~4M params.

In [7]:
head_classes = ({name: l["num_classes"] for name, l in langs.items()}
                if STAGE == "multilingual_pretrain" else {HEAD: num_classes})

model = MultiHeadSignModel(
    in_ch=IN_CH,
    head_classes=head_classes,
    d_model=D_MODEL,
    gcn_hidden=GCN_HIDDEN,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    max_len=MAX_LEN,
).to(device)

# MPS doesn't support the nested-tensor fast path in eval mode; disable it.
for m in model.modules():
    if isinstance(m, nn.TransformerEncoder):
        m.enable_nested_tensor = False
        m.use_nested_tensor = False

if PRETRAINED:
    ckpt = torch.load(PRETRAINED, map_location=device)
    state = ckpt.get("model", ckpt)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded pretrained  missing={len(missing)}  unexpected={len(unexpected)}")
else:
    print("training from scratch")


def unwrap(m):
    """Underlying MultiHeadSignModel, stripping nn.DataParallel's wrapper if
    present. Kept as a no-op passthrough now that multi-GPU is off (single
    GPU / cuda:0 only, see §3) -- avoided the DataParallel misaligned-address
    crash on Kaggle's T4x2 entirely rather than chase it further. .heads /
    .state_dict() / .load_state_dict() calls elsewhere still go through this
    so nothing else needs to change if multi-GPU is ever revisited."""
    return m.module if isinstance(m, nn.DataParallel) else m


print(f"heads: {list(head_classes)}")
print(f"trainable parameters: {count_parameters(unwrap(model)):,}")


training from scratch
heads: ['ksl', 'wlasl']
trainable parameters: 3,899,491


## 8. Optimiser & Scheduler

**Two-phase learning-rate schedule** — critical for Transformer stability:

1. **Warmup** (epochs 1 – `WARMUP_EPOCHS`): LR ramps 10 % → 100 % linearly.  
   Without this, large initial gradients corrupt the attention layers before
   the model has learned useful representations.
2. **Plateau** (after warmup): `ReduceLROnPlateau` halves LR when val accuracy
   stalls for `SCHED_PATIENCE` epochs.

In [8]:
def build_optimizer(model, enc_lr, head_lr, weight_decay):
    """Layer-wise LR: smaller on the (possibly pretrained) encoder, larger on the
    head(s). Works whether there's one head (HEAD) or many (multilingual_pretrain,
    one per language) -- it just unions params across whatever's in model.heads.
    Uses unwrap() so this works whether `model` is DataParallel-wrapped or not
    (DataParallel doesn't expose .heads directly, only via .module)."""
    base        = unwrap(model)
    head_ids    = {id(p) for head in base.heads.values() for p in head.parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids     and p.requires_grad]
    enc_params  = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    return torch.optim.AdamW(
        [{"params": enc_params,  "lr": enc_lr},
         {"params": head_params, "lr": head_lr}],
        weight_decay=weight_decay,
    )


def cycle_loader(loader):
    """Infinite generator over a DataLoader -- reshuffles every full pass (each
    `for batch in loader` re-triggers loader.__iter__()). Used by the
    multilingual_pretrain round-robin loop in §9."""
    while True:
        for batch in loader:
            yield batch


optim = build_optimizer(model, ENC_LR, HEAD_LR, WEIGHT_DECAY)

# Phase 1: linear warm-up (Transformer needs stable early gradients)
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optim, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
# Phase 2: halve LR when val accuracy stops improving
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optim, mode="max", factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-6,
)
print(f"warmup {WARMUP_EPOCHS} epochs → plateau patience {SCHED_PATIENCE}")


warmup 10 epochs → plateau patience 12


### Resume from checkpoint (optional)

Run this cell **instead of** the Optimiser cell when continuing a previous run.
It restores model weights, optimizer momentum, and scheduler state exactly.

In [9]:
RESUME_PATH = str(Path(OUT_PATH).parent / f"{STAGE}_resume.pt")

if Path(RESUME_PATH).exists():
    ckpt = torch.load(RESUME_PATH, map_location=device)
    unwrap(model).load_state_dict(ckpt["model"])

    optim = build_optimizer(model, ENC_LR, HEAD_LR, WEIGHT_DECAY)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optim, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optim, mode="max", factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-6,
    )
    optim.load_state_dict(ckpt["optimizer"])
    # optimizer.load_state_dict() restores the OLD param_groups' weight_decay from
    # the checkpoint too -- re-apply the current config so a WEIGHT_DECAY change
    # actually takes effect on resume instead of silently reverting. LR is left
    # alone; it's still owned by the warmup/plateau schedulers restored below.
    for pg in optim.param_groups:
        pg["weight_decay"] = WEIGHT_DECAY
    warmup_sched.load_state_dict(ckpt["warmup"])
    plateau_sched.load_state_dict(ckpt["plateau"])

    START_EPOCH = ckpt["epoch"] + 1
    history     = ckpt["history"]
    if STAGE == "multilingual_pretrain":
        best_avg_acc = ckpt["best_avg_acc"]
        print(f"resumed from epoch {ckpt['epoch']}  best_avg_acc={best_avg_acc:.3f}")
    else:
        best_acc = ckpt["best_acc"]
        print(f"resumed from epoch {ckpt['epoch']}  best_acc={best_acc:.3f}")
else:
    START_EPOCH = 1
    if STAGE == "multilingual_pretrain":
        best_avg_acc = 0.0
        history      = {"avg_val_acc": []}
    else:
        best_acc = 0.0
        history  = {"tr_loss": [], "tr_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    print("no resume checkpoint found -- starting from scratch")


resumed from epoch 355  best_avg_acc=0.567


## 9. Training

Best checkpoint (by val accuracy) saved to  automatically.

In [10]:
@torch.no_grad()
def evaluate(model, loader, device):
    # NOTE: no label_smoothing here on purpose -- val loss must stay plain
    # cross-entropy so a rising val loss is still a trustworthy overfit signal.
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    for feats, vis, fmask, labels in loader:
        feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
        logits = model(feats, vis, fmask, head=HEAD)
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += labels.numel()
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate_all(model, langs, device):
    """multilingual_pretrain only: evaluate() per language against its own
    val loader/head. Returns {lang: (loss, acc)}."""
    model.eval()
    results = {}
    for name, lang in langs.items():
        correct = total = 0
        loss_sum = 0.0
        for feats, vis, fmask, labels in lang["val_loader"]:
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
            logits = model(feats, vis, fmask, head=name)
            loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            correct  += (logits.argmax(1) == labels).sum().item()
            total    += labels.numel()
        results[name] = (loss_sum / max(total, 1), correct / max(total, 1))
    return results


In [11]:
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
CKPT_DIR = Path(OUT_PATH).parent

if STAGE == "multilingual_pretrain":
    lang_names = sorted(langs)                      # fixed round-robin order
    train_iters = {name: cycle_loader(l["train_loader"]) for name, l in langs.items()}
    steps_per_epoch = STEPS_PER_LANG * len(lang_names)

    epoch_bar = tqdm(range(START_EPOCH, EPOCHS + 1), desc=STAGE, unit="epoch")
    for epoch in epoch_bar:
        model.train()
        run = {name: {"loss": 0.0, "correct": 0, "total": 0} for name in lang_names}

        for step in range(steps_per_epoch):
            name = lang_names[step % len(lang_names)]
            feats, vis, fmask, labels = next(train_iters[name])
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))

            logits = model(feats, vis, fmask, head=name)
            loss   = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTHING)

            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

            run[name]["loss"]    += loss.item() * labels.numel()
            run[name]["correct"] += (logits.argmax(1) == labels).sum().item()
            run[name]["total"]   += labels.numel()

        val_results = evaluate_all(model, langs, device)
        avg_val_acc = sum(acc for _, acc in val_results.values()) / len(val_results)

        if epoch <= WARMUP_EPOCHS:
            warmup_sched.step()
        else:
            plateau_sched.step(avg_val_acc)
        current_lr = optim.param_groups[0]["lr"]

        history["avg_val_acc"].append(avg_val_acc)
        flag = ""
        if avg_val_acc > best_avg_acc:
            best_avg_acc = avg_val_acc
            arch_cfg = dict(in_ch=IN_CH, d_model=D_MODEL, gcn_hidden=GCN_HIDDEN,
                            nhead=NHEAD, num_layers=NUM_LAYERS, dim_ff=DIM_FF,
                            dropout=DROPOUT, max_len=MAX_LEN)
            torch.save(
                {"model": unwrap(model).state_dict(),
                 "class_to_idx": {name: l["class_to_idx"] for name, l in langs.items()},
                 "avg_val_acc": avg_val_acc, "per_language_val": val_results,
                 "epoch": epoch, "arch": arch_cfg},
                OUT_PATH,
            )
            flag = "  <- best, saved"

        # Resume checkpoint (fixed name per stage -- always overwritten)
        torch.save({
            "epoch":        epoch,
            "model":        unwrap(model).state_dict(),
            "optimizer":    optim.state_dict(),
            "warmup":       warmup_sched.state_dict(), 
            "plateau":      plateau_sched.state_dict(),
            "best_avg_acc": best_avg_acc,
            "history":      history,
        }, CKPT_DIR / f"{STAGE}_resume.pt")

        epoch_bar.set_postfix(
            lr=f"{current_lr:.1e}", avg_val=f"{avg_val_acc:.3f}", best=f"{best_avg_acc:.3f}",
        )
        per_lang_str = "  ".join(
            f"{name}={run[name]['correct']/max(run[name]['total'],1):.2f}/"
            f"{val_results[name][1]:.2f}"
            for name in lang_names)
        print(f"epoch {epoch:>3}  lr {current_lr:.2e}  avg_val_acc {avg_val_acc:.3f}{flag}")
        print(f"    (train/val acc per language)  {per_lang_str}")

    print(f"\nbest avg val acc: {best_avg_acc:.4f}  ->  {OUT_PATH}")

else:
    history = {"tr_loss": [], "tr_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_acc = 0.0
    best_path = None

    START_EPOCH = locals().get("START_EPOCH", 1)
    epoch_bar = tqdm(range(START_EPOCH, EPOCHS + 1), desc=f"{STAGE}", unit="epoch")

    for epoch in epoch_bar:
        model.train()
        run_loss = run_correct = run_total = 0

        for feats, vis, fmask, labels in train_loader:
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
            logits = model(feats, vis, fmask, head=HEAD)
            loss   = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTHING)

            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

            run_loss    += loss.item() * labels.numel()
            run_correct += (logits.argmax(1) == labels).sum().item()
            run_total   += labels.numel()

        tr_loss = run_loss    / run_total
        tr_acc  = run_correct / run_total
        val_loss, val_acc = evaluate(model, val_loader, device)

        if epoch <= WARMUP_EPOCHS:
            warmup_sched.step()
        else:
            plateau_sched.step(val_acc)

        current_lr = optim.param_groups[0]["lr"]
        history["tr_loss"].append(tr_loss)
        history["tr_acc"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        flag = ""
        if val_acc > best_acc:
            best_acc  = val_acc
            save_name = f"{STAGE}_e{epoch:03d}_acc{val_acc:.4f}.pt"
            save_path = CKPT_DIR / save_name
            arch_cfg  = dict(in_ch=IN_CH, d_model=D_MODEL, gcn_hidden=GCN_HIDDEN,
                             nhead=NHEAD, num_layers=NUM_LAYERS, dim_ff=DIM_FF,
                             dropout=DROPOUT, max_len=MAX_LEN)
            torch.save(
                {"model": unwrap(model).state_dict(), "class_to_idx": class_to_idx,
                 "val_acc": val_acc, "epoch": epoch, "arch": arch_cfg},
                save_path,
            )
            # Also refresh OUT_PATH (the fixed name other stages/scripts load via PRETRAINED)
            torch.save(
                {"model": unwrap(model).state_dict(), "class_to_idx": class_to_idx,
                 "val_acc": val_acc, "epoch": epoch, "arch": arch_cfg},
                OUT_PATH,
            )
            best_path = save_path
            flag = f"  <- best  ({save_name})"

        # Resume checkpoint (fixed name per stage -- always overwritten)
        torch.save({
            "epoch":        epoch,
            "model":        unwrap(model).state_dict(),
            "optimizer":    optim.state_dict(),
            "warmup":       warmup_sched.state_dict(),
            "plateau":      plateau_sched.state_dict(),
            "best_acc":     best_acc,
            "history":      history,
            "class_to_idx": class_to_idx,
        }, CKPT_DIR / f"{STAGE}_resume.pt")

        epoch_bar.set_postfix(
            lr=f"{current_lr:.1e}",
            tr=f"{tr_acc:.3f}",
            val=f"{val_acc:.3f}",
            best=f"{best_acc:.3f}",
        )
        print(f"epoch {epoch:>3}  lr {current_lr:.2e}  "
              f"train loss {tr_loss:.3f}  acc {tr_acc:.3f}  |  "
              f"val loss {val_loss:.3f}  acc {val_acc:.3f}{flag}")

    print(f"\nbest val acc: {best_acc:.4f}  ->  {best_path}")


multilingual_pretrain:   0%|          | 0/145 [00:00<?, ?epoch/s]

multilingual_pretrain:   1%|          | 1/145 [00:46<1:50:27, 46.02s/epoch, avg_val=0.563, best=0.567, lr=1.6e-05]

epoch 356  lr 1.56e-05  avg_val_acc 0.563
    (train/val acc per language)  ksl=0.90/0.88  wlasl=0.59/0.25


multilingual_pretrain:   1%|▏         | 2/145 [01:27<1:43:11, 43.29s/epoch, avg_val=0.560, best=0.567, lr=7.8e-06]

epoch 357  lr 7.81e-06  avg_val_acc 0.560
    (train/val acc per language)  ksl=0.90/0.88  wlasl=0.58/0.24


multilingual_pretrain:   2%|▏         | 3/145 [02:06<1:37:56, 41.38s/epoch, avg_val=0.561, best=0.567, lr=7.8e-06]

epoch 358  lr 7.81e-06  avg_val_acc 0.561
    (train/val acc per language)  ksl=0.89/0.88  wlasl=0.60/0.24


multilingual_pretrain:   2%|▏         | 3/145 [02:16<1:48:04, 45.66s/epoch, avg_val=0.561, best=0.567, lr=7.8e-06]


KeyboardInterrupt: 

## 10. Results

In [ ]:
if STAGE == "multilingual_pretrain":
    epochs_x = range(1, len(history["avg_val_acc"]) + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs_x, history["avg_val_acc"])
    plt.title("Average val accuracy across languages")
    plt.xlabel("epoch"); plt.ylabel("avg val acc")
    plt.tight_layout()
    plt.show()
    print(f"best avg val acc: {best_avg_acc:.3f}")

else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs_x = range(1, len(history["tr_loss"]) + 1)

    axes[0].plot(epochs_x, history["tr_loss"], label="train")
    axes[0].plot(epochs_x, history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

    axes[1].plot(epochs_x, history["tr_acc"], label="train")
    axes[1].plot(epochs_x, history["val_acc"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()

    axes[2].plot(epochs_x, history["lr"])
    axes[2].set_title("Learning Rate"); axes[2].set_xlabel("epoch")
    axes[2].set_yscale("log")

    plt.tight_layout()
    plt.show()
    print(f"best val acc: {best_acc:.3f}")


### collect predictions + top-k + per-class P/R/F1:

In [ ]:
import numpy as np 

def _topk_accuracy(logits, labels, k_list):
    """logits (N,C), labels (N,) -> {k: accuracy}. k values >= num_classes are
    dropped (trivially 100%, not informative)."""
    num_classes = logits.shape[1]
    k_list = sorted({k for k in k_list if k < num_classes} | {1})
    out = {}
    for k in k_list:
        topk = logits.topk(k, dim=1).indices
        out[k] = (topk == labels.unsqueeze(1)).any(1).float().mean().item()
    return out


def _confusion_prf1(preds, labels, num_classes):
    """preds, labels: 1D int numpy arrays. Returns precision, recall, f1,
    support, cm -- cm (num_classes, num_classes) is the full confusion matrix
    (row = true class, col = predicted class); the rest are derived from its
    diagonal/row-sums/col-sums via a vectorised bincount."""
    idx = labels.astype(np.int64) * num_classes + preds.astype(np.int64)
    cm = np.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    tp = np.diag(cm).astype(float)
    support = cm.sum(axis=1).astype(float)       # true count per class
    pred_count = cm.sum(axis=0).astype(float)     # predicted count per class
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros_like(tp), where=denom > 0)
    return precision, recall, f1, support, cm


@torch.no_grad()
def collect_eval(model, loader, head, idx_to_class, device, k_list=(1, 3, 5)):
    """Run eval-mode model over one loader/head. Returns a dict with top-k
    accuracy plus per-class precision/recall/f1/support (aligned to class
    names via idx_to_class)."""
    model.eval()
    all_logits, all_labels = [], []
    for feats, vis, fmask, labels in loader:
        feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
        logits = model(feats, vis, fmask, head=head)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    num_classes = all_logits.shape[1]

    topk_acc = _topk_accuracy(all_logits, all_labels, k_list)
    preds = all_logits.argmax(1).numpy()
    precision, recall, f1, support, cm = _confusion_prf1(preds, all_labels.numpy(), num_classes)
    names = [idx_to_class[i] for i in range(num_classes)]
    return dict(topk_acc=topk_acc, precision=precision, recall=recall,
               f1=f1, support=support, names=names, cm=cm)


if STAGE == "multilingual_pretrain":
    eval_results = {
        name: collect_eval(model, l["val_loader"], name,
                           {v: k for k, v in l["class_to_idx"].items()}, device)
        for name, l in langs.items()
    }
else:
    eval_results = {
        HEAD: collect_eval(model, val_loader, HEAD,
                           {v: k for k, v in class_to_idx.items()}, device)
    }

for name, r in eval_results.items():
    topk_str = "  ".join(f"top{k}={acc:.3f}" for k, acc in r["topk_acc"].items())
    print(f"[{name}] {topk_str}  |  {len(r['names'])} classes")


### top-k accuracy bar chart

In [ ]:
fig, axes = plt.subplots(1, len(eval_results), figsize=(4 * len(eval_results), 4), squeeze=False)
for ax, (name, r) in zip(axes[0], eval_results.items()):
    ks = list(r["topk_acc"].keys())
    accs = [r["topk_acc"][k] for k in ks]
    ax.bar([f"top-{k}" for k in ks], accs, color="#4C72B0")
    ax.set_ylim(0, 1)
    ax.set_title(name)
    for i, a in enumerate(accs):
        ax.text(i, a + 0.02, f"{a:.2f}", ha="center", fontsize=9)
plt.suptitle("Top-k accuracy")
plt.tight_layout()
plt.show()


### per-class precision/recall/F1, worst-first

In [ ]:
# Full class list if it fits on one readable chart, otherwise just the worst
# MAX_CLASSES_SHOWN by F1 -- e.g. WLASL's 2000 classes would be unreadable
# (and mostly uninteresting) as one giant chart.
MAX_CLASSES_SHOWN = 50

for name, r in eval_results.items():
    order = np.argsort(r["f1"])                    # ascending -> worst first
    n_show = min(len(order), MAX_CLASSES_SHOWN)
    shown = order[:n_show]
    truncated = n_show < len(order)

    fig, ax = plt.subplots(figsize=(8, max(3, n_show * 0.22)))
    y = np.arange(n_show)
    ax.barh(y - 0.25, r["precision"][shown], height=0.25, label="precision", color="#4C72B0")
    ax.barh(y,        r["recall"][shown],    height=0.25, label="recall",    color="#DD8452")
    ax.barh(y + 0.25, r["f1"][shown],        height=0.25, label="f1",        color="#55A868")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['names'][i]} (n={int(r['support'][i])})" for i in shown], fontsize=8)
    ax.invert_yaxis()                               # worst class at the top
    ax.set_xlim(0, 1)
    title = f"{name} -- per-class P/R/F1"
    if truncated:
        title += f"  (worst {n_show}/{len(order)} by F1)"
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()


### top confusions + confusion sub-matrix

`eval_results[name]["cm"]` (added above) is the full (num_classes, num_classes)
confusion matrix. A full heatmap of it is unreadable once a head has more than
a couple dozen classes (KSL alone has ~190), so instead:

1. list the top individual (true -> predicted) confusion pairs by raw count
2. plot a small heatmap restricted to just the classes involved in those
   pairs -- readable regardless of how many classes the head actually has.


In [ ]:
TOP_CONFUSIONS_N = 20        # (true, predicted) pairs listed per head
TOP_CONFUSION_CLASSES = 20   # classes shown in the confusion sub-matrix heatmap


def _top_confusions(cm, names, top_n=20):
    """cm: (C, C) int array (row=true, col=predicted). Returns the top_n
    off-diagonal (true, predicted) cells by raw count, as
    (true_name, pred_name, count, rate_of_true_class) -- the diagonal
    (correct predictions) is excluded since those aren't confusions."""
    c = cm.copy()
    np.fill_diagonal(c, 0)
    support = cm.sum(axis=1)                       # true-class totals
    flat_order = np.argsort(c.ravel())[::-1]
    rows = []
    for flat_idx in flat_order[:top_n]:
        i, j = divmod(flat_idx, cm.shape[1])
        count = int(c[i, j])
        if count <= 0:
            break
        rate = count / support[i] if support[i] > 0 else 0.0
        rows.append((names[i], names[j], count, rate))
    return rows


confusion_tables = {name: _top_confusions(r["cm"], r["names"], TOP_CONFUSIONS_N)
                    for name, r in eval_results.items()}

for name, pairs in confusion_tables.items():
    print(f"[{name}] top confusions (true -> predicted : count, % of that true class):")
    if not pairs:
        print("    (no misclassifications in the val set)")
    for true_c, pred_c, count, rate in pairs:
        print(f"    {true_c!r:>28} -> {pred_c!r:<28} {count:>4}  ({rate:.1%})")
    print()


In [ ]:
for name, r in eval_results.items():
    pairs = confusion_tables[name]
    if not pairs:
        continue

    # Classes involved in the top confusions, first-seen order, capped so the
    # heatmap stays readable.
    involved = []
    for true_c, pred_c, _, _ in pairs:
        for c_ in (true_c, pred_c):
            if c_ not in involved:
                involved.append(c_)
    involved = involved[:TOP_CONFUSION_CLASSES]
    idxs = [r["names"].index(c_) for c_ in involved]
    sub = r["cm"][np.ix_(idxs, idxs)]

    fig, ax = plt.subplots(figsize=(0.5 * len(idxs) + 2, 0.5 * len(idxs) + 2))
    im = ax.imshow(sub, cmap="Blues")
    ax.set_xticks(range(len(idxs)))
    ax.set_xticklabels(involved, rotation=90, fontsize=8)
    ax.set_yticks(range(len(idxs)))
    ax.set_yticklabels(involved, fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    vmax = sub.max() if sub.size else 0
    for i in range(len(idxs)):
        for j in range(len(idxs)):
            v = sub[i, j]
            if v > 0:
                ax.text(j, i, str(int(v)), ha="center", va="center", fontsize=7,
                        color="white" if v > vmax / 2 else "black")
    ax.set_title(f"{name} -- confusion sub-matrix (top {len(idxs)} most-confused classes)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


# Saving Training Process Data to Excel File

In [ ]:
import pandas as pd

# One sheet per language/head (per-class precision/recall/f1/support, worst-
# first), one "topk_summary" sheet, and one "training_history" sheet (the
# epoch-by-epoch train/val loss+acc+lr curves, or avg_val_acc for
# multilingual_pretrain) -- lets you re-plot or filter freely in Excel
# without re-running the notebook.
EVAL_XLSX_PATH = Path(f"../../plots/eval_metrics_{STAGE}.xlsx")
EVAL_XLSX_PATH.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(EVAL_XLSX_PATH, engine="openpyxl") as writer:
    summary_rows = []
    for name, r in eval_results.items():
        df = pd.DataFrame({
            "class": r["names"],
            "support": r["support"].astype(int),
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
        }).sort_values("f1")
        sheet = name[:31]   # Excel sheet-name length limit
        df.to_excel(writer, sheet_name=sheet, index=False)
        for k, acc in r["topk_acc"].items():
            summary_rows.append({"language": name, "k": k, "top_k_accuracy": acc})
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="topk_summary", index=False)

    for name, pairs in confusion_tables.items():
        if not pairs:
            continue
        conf_df = pd.DataFrame(pairs, columns=["true_class", "predicted_class",
                                               "count", "rate_of_true_class"])
        sheet = f"{name[:20]}_confusions"[:31]   # Excel sheet-name length limit
        conf_df.to_excel(writer, sheet_name=sheet, index=False)

    # history's keys differ by stage (full tr/val loss+acc+lr vs. just
    # avg_val_acc for multilingual_pretrain) -- pd.DataFrame handles either
    # shape directly since every value in `history` is an equal-length list.
    hist_df = pd.DataFrame(history)
    hist_df.insert(0, "epoch", range(1, len(hist_df) + 1))
    hist_df.to_excel(writer, sheet_name="training_history", index=False)

print(f"saved eval metrics + training history -> {EVAL_XLSX_PATH.resolve()}")
